# DocFinder — Indexeur Colab (GPU)

Ce notebook tourne sur Colab GPU et prend en charge tout le travail lourd d'embedding.

## Architecture
```
Mac (local) ──── /chunks NDJSON ───► Colab GPU ──── embeddings ───► Mac /admin/upsert ───► Qdrant local
              (extraction texte)    (sentence-transformers)         (FastAPI port 8000)    (port 6333)
```

Colab ne se connecte **jamais** directement à Qdrant — tout passe par le serveur FastAPI du Mac via Cloudflare Tunnel.

## Flux automatique
1. Colab poll `/admin/ping` toutes les 5s via le tunnel
2. Quand `status=running` → fetch `/chunks` en streaming
3. Calcule les embeddings dense sur GPU (FP16 si CUDA)
4. POST les points vers `/admin/upsert` sur le Mac → le Mac écrit dans Qdrant local

## Prérequis
- Serveur FastAPI en cours d'exécution sur le Mac : `bash start.sh`
- **Un seul tunnel Cloudflare** suffit : `cloudflared tunnel --url http://localhost:8000`
  L'URL affichée (ex. `https://xxx.trycloudflare.com`) est **stable** entre les restarts

## Démarrage
1. Renseignez `DOCFINDER_URL` dans la cellule **Config** ci-dessous
2. **Runtime → Run all** — le daemon démarre automatiquement à la dernière cellule
3. Cliquez **Lancer l'indexation** dans http://localhost:8000/admin

In [ ]:
# ── Installation des dépendances ──────────────────────────────────────────────
# sentence-transformers charge le modèle intfloat/multilingual-e5-large
# requests gère les appels HTTP vers le Mac via Cloudflare Tunnel
!pip install -q sentence-transformers requests

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Config — seule cellule à modifier
# ─────────────────────────────────────────────────────────────────────────────

# URL Cloudflare Tunnel du Mac (port 8000)
# Sur le Mac : cloudflared tunnel --url http://localhost:8000
# → copiez l'URL HTTPS affichée (ex. https://xxx.trycloudflare.com)
# L'URL est stable entre les restarts — pas besoin de la changer régulièrement.
DOCFINDER_URL = "https://routers-prompt-designed-dirt.trycloudflare.com"

# Modèle d'embedding (doit être identique au serveur local)
# multilingual-e5-large : 1024 dims, bien supérieur à mpnet pour le français.
# Requiert les préfixes "passage: " (indexation) et "query: " (recherche).
MODEL_NAME = "intfloat/multilingual-e5-large"

# Nombre de points accumulés avant chaque envoi au Mac
# Valeur basse = Qdrant se met à jour plus souvent, progression visible en temps réel
UPSERT_EVERY = 50

# Intervalle de polling du daemon (secondes)
POLL_INTERVAL = 5

# ── Normalisation + validation immédiate ─────────────────────────────────────
DOCFINDER_URL = DOCFINDER_URL.strip().rstrip("/")   # supprime espaces et slash final
if not DOCFINDER_URL.startswith("https://") or "xxx" in DOCFINDER_URL:
    raise ValueError(
        "⛔  Mettez à jour DOCFINDER_URL avant de continuer.\n"
        "    Sur le Mac : cloudflared tunnel --url http://localhost:8000\n"
        "    → copiez l'URL HTTPS affichée (ex. https://abc123.trycloudflare.com)"
    )

print(f"DocFinder URL : {DOCFINDER_URL}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Setup : GPU, modèle, connexion Mac
# ─────────────────────────────────────────────────────────────────────────────
import time
import requests
import torch
from sentence_transformers import SentenceTransformer

# Cloudflare Tunnel ne nécessite pas de headers spéciaux (contrairement à ngrok).
_HEADERS: dict = {}

# ── Détection GPU et réglage automatique du BATCH_SIZE ───────────────────────
# Le BATCH_SIZE contrôle combien de chunks sont encodés simultanément.
# Valeurs choisies pour occuper ~80% de la VRAM sans OOM sur les modèles courants.
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

    if vram_gb >= 75:      # A100 80 GB
        BATCH_SIZE = 256
    elif vram_gb >= 20:    # L4 (22.5 GB), A10G (23.7 GB)
        BATCH_SIZE = 128
    elif vram_gb >= 14:    # T4 15.8 GB (Colab gratuit)
        BATCH_SIZE = 64
    else:                  # GPU modeste (< 14 GB)
        BATCH_SIZE = 32

    device = "cuda"
    print(f"GPU : {gpu_name} ({vram_gb:.1f} GB VRAM) → BATCH_SIZE={BATCH_SIZE}")
else:
    device     = "cpu"
    BATCH_SIZE = 16
    gpu_name   = "CPU"
    print(f"Aucun GPU — CPU seul, BATCH_SIZE={BATCH_SIZE}")

# ── Chargement du modèle ──────────────────────────────────────────────────────
model = SentenceTransformer(MODEL_NAME, device=device)

if device == "cuda":
    # FP16 : réduit la VRAM de moitié et accélère l'encodage d'environ 2x sur T4/A100.
    # La perte de précision est négligeable pour la similarité cosinus.
    model.half()
    print(f"FP16 activé — encodage ~2× plus rapide sur {gpu_name}")

print(f"Modèle '{MODEL_NAME}' prêt.")

# ── Test de connexion au Mac ──────────────────────────────────────────────────
try:
    r = requests.get(f"{DOCFINDER_URL}/health", headers=_HEADERS, timeout=15)
    r.raise_for_status()
    health = r.json()
    assert health.get("status") == "ok", f"Réponse inattendue : {health}"
    print(f"Connexion Mac OK ✓  ({health})")
except requests.exceptions.ConnectionError:
    raise SystemError(
        "⛔  Impossible de joindre le Mac.\n"
        "    Vérifiez que :\n"
        "    1. Le serveur FastAPI tourne (bash start.sh sur le Mac)\n"
        "    2. Le tunnel Cloudflare tourne (cloudflared tunnel --url http://localhost:8000)\n"
        "    3. DOCFINDER_URL correspond bien à l'URL affichée par cloudflared"
    )
except requests.exceptions.HTTPError as exc:
    raise SystemError(
        f"⛔  Le Mac a répondu avec une erreur HTTP : {exc}\n"
        "    Vérifiez que le serveur FastAPI est bien démarré."
    )

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Helpers réseau : retry HTTP, heartbeat, control, stream producteur
# ─────────────────────────────────────────────────────────────────────────────
import json
import queue
import threading

# Délais du backoff exponentiel (secondes) — 5 tentatives, cap à 16s.
# Couvre les coupures réseau transitoires (idle timeout, tunnel restart).
_RETRY_DELAYS = (1, 2, 4, 8, 16)

# Nombre max de batches en avance dans la queue producteur → consommateur.
# Limite la consommation mémoire si l'encodage GPU est plus rapide que le réseau.
QUEUE_MAXSIZE = 4

# Nombre de fichiers skippés accumulés avant d'envoyer un report au Mac.
# Évite de faire un appel HTTP par fichier tout en gardant l'UI réactive.
_SKIP_REPORT_EVERY = 10


def _http_get(url: str, **kwargs) -> requests.Response:
    """GET avec retry exponentiel sur ConnectionError, Timeout et 502/503/504."""
    for attempt, delay in enumerate(_RETRY_DELAYS, 1):
        try:
            r = requests.get(url, headers=_HEADERS, **kwargs)
            if r.status_code in (502, 503, 504):
                raise requests.exceptions.ConnectionError(f"HTTP {r.status_code}")
            return r
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as exc:
            if attempt == len(_RETRY_DELAYS):
                raise
            print(f"  [retry {attempt}/{len(_RETRY_DELAYS)}] {exc} → dans {delay}s")
            time.sleep(delay)
    raise RuntimeError("unreachable")


def _http_post(url: str, **kwargs) -> requests.Response:
    """POST avec retry exponentiel — même logique que _http_get."""
    for attempt, delay in enumerate(_RETRY_DELAYS, 1):
        try:
            r = requests.post(url, headers=_HEADERS, **kwargs)
            if r.status_code in (502, 503, 504):
                raise requests.exceptions.ConnectionError(f"HTTP {r.status_code}")
            return r
        except (requests.exceptions.ConnectionError, requests.exceptions.Timeout) as exc:
            if attempt == len(_RETRY_DELAYS):
                raise
            print(f"  [retry {attempt}/{len(_RETRY_DELAYS)}] {exc} → dans {delay}s")
            time.sleep(delay)
    raise RuntimeError("unreachable")


def get_status() -> dict:
    """Interroge /admin/ping sur le Mac et retourne le statut du job courant."""
    r = _http_get(f"{DOCFINDER_URL}/admin/ping", timeout=30)
    return r.json()


def send_heartbeat() -> None:
    """Envoie un heartbeat au Mac — fire-and-forget."""
    try:
        requests.post(
            f"{DOCFINDER_URL}/admin/colab/heartbeat",
            headers=_HEADERS,
            json={"device": device, "gpu": gpu_name},
            timeout=5,
        )
    except Exception:
        pass


def check_paused() -> bool:
    """Retourne True si le Mac a demandé une pause — best-effort, sans retry."""
    try:
        r = requests.get(
            f"{DOCFINDER_URL}/admin/colab/control",
            headers=_HEADERS,
            timeout=5,
        )
        return r.json().get("paused", False)
    except Exception:
        return False


def fetch_indexed_doc_ids() -> set:
    """Récupère les doc_id déjà présents dans Qdrant pour permettre la reprise."""
    try:
        r = _http_get(f"{DOCFINDER_URL}/admin/indexed-doc-ids", timeout=30)
        ids = set(r.json().get("doc_ids", []))
        if ids:
            print(f"  Reprise : {len(ids)} docs déjà indexés — ignorés.")
        return ids
    except Exception as exc:
        print(f"  Avertissement : impossible de récupérer les doc_ids existants ({exc})")
        return set()


def _report_skipped(count: int) -> None:
    """Envoie au Mac le nombre de fichiers skippés pour mise à jour de l'UI — best-effort."""
    if count <= 0:
        return
    try:
        _http_post(
            f"{DOCFINDER_URL}/admin/colab/skip",
            json={"count": count},
            timeout=10,
        )
    except Exception:
        pass


def _stream_producer(path, batch_q: queue.Queue, skip_doc_ids: set) -> None:
    """Thread producteur : consomme le flux NDJSON /chunks et remplit la queue.

    Items déposés dans la queue (tous sous forme de dict typés) :
      {"type": "meta",      "total_files": N}         — nb fichiers total
      {"type": "file",      "path": ..., "chunks": N} — fichier avec nouveaux chunks
      {"type": "skip_file", "path": ...}               — fichier entièrement déjà indexé
      {"type": "batch",     "data": [...]}              — chunks à encoder
      None                                              — sentinel de fin

    Le buffering de "pending_file" garantit qu'un événement "file" n'est émis
    que lorsqu'au moins un chunk non-skippé est trouvé pour ce fichier.
    Sinon un "skip_file" est émis à la place, pour maintenir la barre de progression.
    """
    url = f"{DOCFINDER_URL}/chunks"
    if path:
        url += f"?path={requests.utils.quote(path)}"
    batch: list = []
    skipped_chunks = 0
    pending_file = None   # buffer : on attend le 1er chunk non-skippé avant d'émettre "file"
    try:
        with requests.get(url, headers=_HEADERS, stream=True, timeout=300) as resp:
            resp.raise_for_status()
            for raw_line in resp.iter_lines():
                if not raw_line:
                    continue
                line = json.loads(raw_line)
                t = line.get("type")
                if t == "meta":
                    batch_q.put({"type": "meta", "total_files": line["total_files"]})
                elif t == "file":
                    # Flush le fichier précédent comme skip_file si aucun chunk non-skippé
                    if pending_file is not None:
                        batch_q.put({"type": "skip_file", "path": pending_file["path"]})
                    pending_file = {"type": "file", "path": line["path"], "chunks": line.get("chunks", 0)}
                elif t == "skip":
                    # Le serveur a skippé ce fichier (vide/illisible) — flush comme skip_file
                    if pending_file is not None:
                        batch_q.put({"type": "skip_file", "path": pending_file["path"]})
                        pending_file = None
                elif t == "chunk":
                    if line.get("doc_id") in skip_doc_ids:
                        skipped_chunks += 1
                        continue
                    # Premier chunk non-skippé pour ce fichier → émettre le "file" bufferisé
                    if pending_file is not None:
                        batch_q.put(pending_file)
                        pending_file = None
                    batch.append(line)
                    if len(batch) >= BATCH_SIZE:
                        batch_q.put({"type": "batch", "data": batch})
                        batch = []
                elif t == "keepalive":
                    pass  # heartbeat anti-timeout Cloudflare — ignorer
                elif t == "done":
                    # Flush le dernier fichier pending s'il n'a produit aucun chunk
                    if pending_file is not None:
                        batch_q.put({"type": "skip_file", "path": pending_file["path"]})
                        pending_file = None
                    if skipped_chunks:
                        print(f"  Flux terminé. ({skipped_chunks} chunks ignorés — déjà indexés)")
                elif t == "error":
                    print(f"  ERREUR flux : {line.get('error')}")
                    break
        # Sécurité : flush si "done" n'est jamais reçu (connexion coupée)
        if pending_file is not None:
            batch_q.put({"type": "skip_file", "path": pending_file["path"]})
        if batch:
            batch_q.put({"type": "batch", "data": batch})
    except Exception as exc:
        print(f"  ERREUR producteur : {exc}")
    finally:
        batch_q.put(None)  # sentinel de fin


def _upsert_batch(points_data: list) -> None:
    """Envoie les points encodés au Mac via /admin/upsert (avec retry HTTP)."""
    r = _http_post(
        f"{DOCFINDER_URL}/admin/upsert",
        json=points_data,
        timeout=120,
    )
    r.raise_for_status()
    resp = r.json()
    if "error" in resp:
        raise RuntimeError(f"Upsert refusé par le Mac : {resp['error']}")


print("Helpers réseau prêts.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Moteur d'indexation GPU : _encode_safe + index_pipeline
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
from pathlib import Path
from tqdm.notebook import tqdm as tqdm_nb

# multilingual-e5 exige ce préfixe sur les passages lors de l'indexation.
# Le serveur local utilise "query: " pour les recherches (voir shared/embedder.py).
PASSAGE_PREFIX = "passage: "


def _encode_safe(texts: list) -> tuple:
    """Encode les textes sur GPU avec gestion automatique des OOM CUDA.

    En cas de OutOfMemoryError, réduit de moitié le batch_size d'encodage
    et réessaie. Met à jour BATCH_SIZE globalement pour les batches futurs.
    Retourne (embeddings_array, batch_size_effectif).
    """
    global BATCH_SIZE
    bs = BATCH_SIZE
    prefixed = [PASSAGE_PREFIX + t for t in texts]
    while True:
        try:
            embs = model.encode(
                prefixed,
                batch_size=bs,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
            return embs, bs
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            bs = max(1, bs // 2)
            BATCH_SIZE = bs
            print(f"  ⚠ OOM CUDA — BATCH_SIZE réduit à {bs}, retry…")


def index_pipeline(path=None) -> int:
    """Pipeline GPU producer/consumer pour l'indexation de documents.

    Affiche deux barres de progression tqdm :
      - Fichiers  : nb fichiers traités / total détecté par le serveur
      - Points    : nb points (chunks encodés) envoyés à Qdrant

    Architecture :
    - Thread producteur (_stream_producer) : stream NDJSON /chunks → queue
    - Thread principal (consommateur)      : encode sur GPU → accumule → upsert Mac

    Retourne le nombre total de points insérés dans Qdrant.
    """
    skip_doc_ids = fetch_indexed_doc_ids()

    batch_q: queue.Queue = queue.Queue(maxsize=QUEUE_MAXSIZE)
    producer = threading.Thread(
        target=_stream_producer, args=(path, batch_q, skip_doc_ids), daemon=True
    )
    producer.start()

    total_inserted = 0
    pending_points: list = []
    skipped_files = 0   # compteur de fichiers ignorés non encore reportés au Mac

    file_pbar = tqdm_nb(total=0,   desc="Fichiers", unit=" fichier", leave=True)
    pts_pbar  = tqdm_nb(total=None, desc="Points → Qdrant", unit=" pts",   leave=True)

    try:
        while True:
            item = batch_q.get()
            if item is None:   # sentinel : le producteur a terminé
                break

            t = item.get("type")

            # ── Métadonnées du job ─────────────────────────────────────────
            if t == "meta":
                file_pbar.reset(total=item["total_files"])
                continue

            if t == "file":
                file_pbar.update(1)
                file_pbar.set_postfix_str(Path(item["path"]).name[:50])
                continue

            # ── Fichier entièrement déjà indexé ───────────────────────────
            if t == "skip_file":
                file_pbar.update(1)
                file_pbar.set_postfix_str(f"⏭ {Path(item['path']).name[:45]}")
                skipped_files += 1
                if skipped_files % _SKIP_REPORT_EVERY == 0:
                    _report_skipped(_SKIP_REPORT_EVERY)
                continue

            # ── Batch de chunks à encoder (t == "batch") ──────────────────
            batch = item["data"]

            # Pause demandée depuis l'admin Mac
            if check_paused():
                file_pbar.set_description("⏸ En pause")
                while check_paused():
                    time.sleep(3)
                file_pbar.set_description("Fichiers")

            texts = [c["content"] for c in batch]
            embeddings, _ = _encode_safe(texts)

            for chunk, emb in zip(batch, embeddings):
                p = {
                    "id":      chunk["point_id"],
                    "dense":   emb.tolist(),
                    "payload": {
                        "doc_id":      chunk["doc_id"],
                        "chunk_id":    chunk["chunk_id"],
                        "title":       chunk["title"],
                        "path":        chunk["path"],
                        "abs_path":    chunk.get("abs_path", ""),
                        "doc_type":    chunk["doc_type"],
                        "content":     chunk["content"],
                        "keywords":    chunk["keywords"],
                        "chunk_index": chunk["chunk_index"],
                    },
                }
                if chunk.get("sparse_indices"):
                    p["sparse_indices"] = chunk["sparse_indices"]
                    p["sparse_values"]  = chunk["sparse_values"]
                pending_points.append(p)

            del embeddings   # libérer la mémoire GPU immédiatement

            # Flush vers Mac dès qu'on a UPSERT_EVERY points
            if len(pending_points) >= UPSERT_EVERY:
                _upsert_batch(pending_points)
                total_inserted += len(pending_points)
                pts_pbar.update(len(pending_points))
                pts_pbar.set_postfix(total=total_inserted)
                pending_points = []
                torch.cuda.empty_cache()

        # Flush final du reste en attente
        if pending_points:
            _upsert_batch(pending_points)
            total_inserted += len(pending_points)
            pts_pbar.update(len(pending_points))
            pts_pbar.set_postfix(total=total_inserted)

        # Flush des fichiers skippés non encore reportés
        remainder = skipped_files % _SKIP_REPORT_EVERY
        if remainder:
            _report_skipped(remainder)

    finally:
        file_pbar.close()
        pts_pbar.close()

    torch.cuda.empty_cache()
    producer.join()
    return total_inserted


print("Moteur d'indexation prêt.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Daemon — polling automatique
#
# Cette cellule démarre automatiquement lors du « Run All ».
# Elle boucle indéfiniment en attendant un job lancé depuis l'admin Mac.
#
# Arrêt : bouton ■ (interrompre le kernel) ou Ctrl+C.
# ─────────────────────────────────────────────────────────────────────────────
# Garde : vérifie que les cellules précédentes ont été exécutées
if 'send_heartbeat' not in dir():
    raise RuntimeError(
        "⛔  Les helpers ne sont pas chargés.\n"
        "   → Runtime > Run all (Ctrl+F9) pour relancer toutes les cellules dans l'ordre."
    )


print("Daemon démarré — en attente d'un job sur le Mac…")
print(f"Poll toutes les {POLL_INTERVAL}s sur {DOCFINDER_URL}/admin/ping")
print("Lancez une indexation depuis http://localhost:8000/admin")
print("─" * 60)

job_running = False

while True:
    try:
        send_heartbeat()
        status = get_status()
        s = status.get("status")

        if s == "running" and not job_running:
            job_running = True
            job_path = status.get("path") or None
            print(f"\n[{time.strftime('%H:%M:%S')}] Job détecté — path: {job_path}")
            try:
                n = index_pipeline(path=job_path)
                print(f"\n[{time.strftime('%H:%M:%S')}] Terminé — {n} points insérés dans Qdrant.")
            except Exception as exc:
                print(f"[{time.strftime('%H:%M:%S')}] ERREUR pipeline : {exc}")

        elif s in ("done", "error", "idle") and job_running:
            job_running = False
            print(f"[{time.strftime('%H:%M:%S')}] Job {s}. En attente du prochain…")
            print("─" * 60)

        elif s == "running" and job_running:
            done  = status.get("done", 0)
            total = status.get("total", 0)
            pct   = status.get("progress_pct", 0)
            print(f"  Mac : {done}/{total} fichiers ({pct}%)…", end="\r")

    except requests.exceptions.ConnectionError:
        print(f"[{time.strftime('%H:%M:%S')}] Mac inaccessible — retry dans {POLL_INTERVAL}s…")
    except Exception as exc:
        print(f"[{time.strftime('%H:%M:%S')}] Erreur : {exc}")

    time.sleep(POLL_INTERVAL)
